# Problem Set 1: Analysis of racial disparities in felony sentencing, Part 1

0. Load packages and imports

In [2]:
import pandas as pd
import numpy as np
import re

## can add others if you need them

## repeated printouts
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

0.1: Load the data (0 points)

Load the data from `sentencing_asof0405.csv`
- *Notes*: You may receive a warning about mixed data types upon import; feel free to ignore

In [ ]:

#df = pd.read_csv("/Users/VIVIANAPEREZ/Documents/GitHub/QSS20_WI26/problemsets/pset1/pset1_inputdata/sentencing_asof0405.csv")
df = pd.read_csv("pset1_inputdata/sentencing_asof0405.csv")
import os
os.getcwd()

/var/folders/k9/0k40031s4q9btj5wvqclnww00000gn/T/ipykernel_54584/4032024950.py:2: DtypeWarning: Columns (10,11,14,25) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("pset1_inputdata/sentencing_asof0405.csv")


'/Users/vivianaperez/Documents/GitHub/QSS20_WI26/problemsets/pset1'

0.2: Print head, dimensions, info (0 points)

In [4]:
df.head()
df.shape
df.info()

,CASE_ID,CASE_PARTICIPANT_ID,RECEIVED_DATE,OFFENSE_CATEGORY,PRIMARY_CHARGE_FLAG,CHARGE_ID,CHARGE_VERSION_ID,DISPOSITION_CHARGED_OFFENSE_TITLE,CHARGE_COUNT,DISPOSITION_DATE,...,INCIDENT_CITY,INCIDENT_BEGIN_DATE,INCIDENT_END_DATE,LAW_ENFORCEMENT_AGENCY,LAW_ENFORCEMENT_UNIT,ARREST_DATE,FELONY_REVIEW_DATE,FELONY_REVIEW_RESULT,ARRAIGNMENT_DATE,UPDATED_OFFENSE_CATEGORY
0,149765331439,175691153649,8/15/1984 12:00:00 AM,PROMIS Conversion,False,50510112469,116304211997,FIRST DEGREE MURDER,2,12/17/2014 12:00:00 AM,...,NaN,8/9/1984 12:00:00 AM,NaN,CHICAGO POLICE DEPT,NaN,8/15/1984 12:00:00 AM,08/15/1984 12:00:00 AM,Charge(S) Approved,9/21/1984 12:00:00 AM,Homicide
1,149765331439,175691153649,8/15/1984 12:00:00 AM,PROMIS Conversion,False,50510213021,98265074680,HOME INVASION,14,12/17/2014 12:00:00 AM,...,NaN,8/9/1984 12:00:00 AM,NaN,CHICAGO POLICE DEPT,NaN,8/15/1984 12:00:00 AM,08/15/1984 12:00:00 AM,Charge(S) Approved,9/21/1984 12:00:00 AM,Homicide
2,149765331439,175691153649,8/15/1984 12:00:00 AM,PROMIS Conversion,False,50516447217,131972895911,FIRST DEGREE MURDER,4,12/17/2014 12:00:00 AM,...,NaN,8/9/1984 12:00:00 AM,NaN,CHICAGO POLICE DEPT,NaN,8/15/1984 12:00:00 AM,08/15/1984 12:00:00 AM,Charge(S) Approved,9/21/1984 12:00:00 AM,Homicide
3,149765331439,175691153649,8/15/1984 12:00:00 AM,PROMIS Conversion,False,50516497493,131966356472,FIRST DEGREE MURDER,5,12/17/2014 12:00:00 AM,...,NaN,8/9/1984 12:00:00 AM,NaN,CHICAGO POLICE DEPT,NaN,8/15/1984 12:00:00 AM,08/15/1984 12:00:00 AM,Charge(S) Approved,9/21/1984 12:00:00 AM,Homicide
4,149765331439,175691153649,8/15/1984 12:00:00 AM,PROMIS Conversion,False,50516648320,98059642859,HOME INVASION,13,12/17/2014 12:00:00 AM,...,NaN,8/9/1984 12:00:00 AM,NaN,CHICAGO POLICE DEPT,NaN,8/15/1984 12:00:00 AM,08/15/1984 12:00:00 AM,Charge(S) Approved,9/21/1984 12:00:00 AM,Homicide


(248146, 41)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 248146 entries, 0 to 248145
Data columns (total 41 columns):
 #   Column                             Non-Null Count   Dtype  
---  ------                             --------------   -----  
 0   CASE_ID                            248146 non-null  int64  
 1   CASE_PARTICIPANT_ID                248146 non-null  int64  
 2   RECEIVED_DATE                      248146 non-null  object 
 3   OFFENSE_CATEGORY                   248146 non-null  object 
 4   PRIMARY_CHARGE_FLAG                248146 non-null  bool   
 5   CHARGE_ID                          248146 non-null  int64  
 6   CHARGE_VERSION_ID                  248146 non-null  int64  
 7   DISPOSITION_CHARGED_OFFENSE_TITLE  248146 non-null  object 
 8   CHARGE_COUNT                       248146 non-null  int64  
 9   DISPOSITION_DATE                   248146 non-null  object 
 10  DISPOSITION_CHARGED_CHAPTER        248146 non-null  object 
 11  DISPOSITION_CHARGED_ACT            2427

Part 1: data cleaning/interpretation

This data set has 248,146 entries and 41 columns. Most of the clumns are objects, but there are still a couple floats and booleans (and 5 integers). Some of the columns -like CHARGE_DISPOSITION_REASON have many null responses, which means the column might not be of as much use.

## 1.1: Understanding the unit of analysis (5 points)

- Print the number of unique values for the following columns. Do so in a way that avoids copying/pasting code for 
the three:

    - Cases (`CASE_ID`)
    - People in that case (`CASE_PARTICIPANT_ID`)
    - Charges (`CHARGE_ID`)

- Write a couple sentences on the following and show an example of each (e.g., a case involving multiple people):
    
    - Why there are more unique people than unique cases?
    - Why there are more unique charges than unique people?

- Print the mean and median number of charges per case

- Print the mean and median number of participants per case

- Does the data seem to enable us to follow the same defendant across different cases they're charged in? Write 1 sentence in support of your conclusion.


In [5]:
cols = ["CASE_ID" , "CASE_PARTICIPANT_ID" , "CHARGE_ID"]

for col in cols:
    print(col)
    print(df[col].nunique())
# There are more unique people than unique cases because one case could involve multiple people.
# There are more unique charges than unique people because one person could have multiple charges.

charges_per_case = df.groupby("CASE_ID")["CHARGE_ID"].nunique()
print(charges_per_case.mean())
print(charges_per_case.median())

participants_per_case = df.groupby("CASE_ID")["CASE_PARTICIPANT_ID"].nunique()
print(participants_per_case.mean())
print(participants_per_case.median())

# No, the data does not enable us to follow the same defendant across different cases.
# This is because the idenitfier "CASE_PARTICIPANT_ID" corresponds to a defendent within the case,
# but it isn't shared across cases, even if it's the same individual.

CASE_ID
197519
CASE_PARTICIPANT_ID
211977
CHARGE_ID
229015
1.1594580774507768
1.0
1.0731980214561636
1.0


## 1.2.1: Which offense is final? (3 points)

- First, read the data documentation [link](https://datacatalog.cookcountyil.gov/api/views/tg8v-tm6u/files/8597cdda-f7e1-44d1-b0ce-0a4e43f8c980?download=true&filename=CCSAO%20Data%20Glossary.pdf) and summarize in your own words the differences between `OFFENSE_CATEGORY` and `UPDATED_OFFENSE_CATEGORY` 

- Construct an indicator `is_changed_offense` that's True for case-participant-charge observations (rows) where there's a difference between the original charge (offense category) and the most current charge (updated offense category). What are some of the more common changed offenses? (can just print result of sort_values based on original offense category)

- Print one example of a changed offense from one of these categories and comment on what the reason may be


In [6]:
# OFFENSE_CATEGORY represents the more general category of offense before an individual was charged.
# UPDATED_OFFENSE_CATEGORY represents the offense category after the first charge (which may or may not be the same as the above.)

df["is_changed_offense"] = df["OFFENSE_CATEGORY"] != df["UPDATED_OFFENSE_CATEGORY"]

df[df["is_changed_offense"]]["OFFENSE_CATEGORY"].value_counts().sort_values(ascending=False)

print("One example of a frequently changed offense is aggrevated battery.")
# This may be because the orignal charge got changed to a lesser form of battery.

OFFENSE_CATEGORY
PROMIS Conversion               6394
DUI                             3896
UUW - Unlawful Use of Weapon    2155
Other Offense                   2125
Aggravated Battery              1927
                                ... 
Perjury                            4
Prostitution                       3
Compelling Gang Membership         2
Benefit Recipient Fraud            2
Violate Bail Bond                  2
Name: count, Length: 88, dtype: int64

One example of a frequently changed offense is aggrevated battery.


## 1.2.2: Simplifying the charges (5 points)

Using the field (`UPDATED_OFFENSE_CATEGORY`), create a new field, `simplified_offense_derived`, that simplifies the many offense categories into broader buckets using the following process:

First, combine all offenses beginning with "Aggravated" into a single category without that prefix (e.g., Aggravated Battery and Battery just becomes Battery)

Then:
- Combine all offenses with arson into a single arson category (`Arson`)
- Combine all offenses with homicide into a single homicide category (`Homicide`)
- Combine all offenses with vehicle/vehicular in the name into a single vehicle category (`Vehicle-related`)
- Combine all offenses with battery in the name into a single battery category (`Battery`)

Try to do so efficiently (e.g., write a function and apply to a column, rather than edit the variable repeatedly in separate line for each recoded offense)

Print the difference between the # of unique offenses in the original `UPDATED_OFFENSE_CATEGORY` field and the # of unique offenses in your new `simplified_offense_derived` field


In [7]:
df["simplified_offense_derived"] = df["UPDATED_OFFENSE_CATEGORY"]

def simplify_offense(offense):
    # Remove 'Aggravated ' at the start if it exists
    if str(offense).startswith("Aggravated "):
        offense = offense.replace("Aggravated ", "")
    
    # Map to broader categories
    if "Arson" in offense:
        return "Arson"
    elif "Homicide" in offense:
        return "Homicide"
    elif "Vehicle" in offense or "Vehicular" in offense:
        return "Vehicle-related"
    elif "Battery" in offense:
        return "Battery"
    else:
        return offense
    
df["simplified_offense_derived"] = df["simplified_offense_derived"].apply(simplify_offense)

original_unique = df["UPDATED_OFFENSE_CATEGORY"].nunique()
simplified_unique = df["simplified_offense_derived"].nunique()
diff_unique = original_unique - simplified_unique

print(diff_unique)


14


## 1.3: Cleaning additional variables (10 points)

Clean the following variables; make sure to retain the original variable in data and use the derived suffix so it's easier to pull these cleaned out variables later (e.g., `age_derived`) to indicate this was a transformation

- Race: create True/false indicators for `is_black_derived` (Black only or mixed race with hispanic), Non-Black Hispanic, so either hispanic alone or white hispanic (`is_hisp_derived`), White non-hispanic (`is_white_derived`), or none of the above (`is_othereth_derived`)

- Gender: create a boolean true/false indicator for `is_male_derived` (false is female, unknown, or other)

- Age at incident: you notice outliers like 130-year olds. Winsorsize the top 0.01% of values to be equal to the 99.99th percentile value pre-winsorization. Call this `age_derived`

- Create `sentenceymd_derived` that's a version of `SENTENCE_DATE` converted to datetime format. Also create a rounded version, `sentenceym_derived`, that's rounded down to the first day of the month (e.g., `1/5/2016` would become `1/1/2016` and `3/27/2018` would become `3/1/2018`)
    - Hint: all timestamps are midnight so u can strip in conversion. For full credit, before converting, you notice that some of the years have been mistranscribed (e.g., 291X or 221X instead of 201X). Programatically fix those (eg 2914 -> 2014). Even after cleaning, there will still be some that are after the year 2021 that we'll filter out later. For partial credit, you can ignore the timestamps that cause errors and set errors = "coerce" within `pd.to_datetime()` to allow the conversion to proceed. 

- Sentencing judge: create an identifier (`judgeid_derived`) for each unique judge (`SENTENCE_JUDGE`) structured as judge_1, judge_2...., with the order determined by sorting the judges (will sort on fname then last). When finding unique judges, there are various duplicates we could weed out --- for now, just focus on (1) the different iterations of Doug/Douglas Simpson, (2) the different iterations of Shelley Sutker (who appears both with her maiden name and her hyphenated married name). 
     - Hint: due to mixed types, you may need to cast the `SENTENCE_JUDGE` var to a diff type to sort

After finishing, print a random sample of 10 rows (data.sample(n = 10)) with the original and cleaned columns for the relevant variables to validate your work

In [12]:
#race
lowercase_race = df["RACE"].str.lower()

df["is_black_derived"] = lowercase_race.str.contains("black", na=False)

df["is_hisp_derived"] = (
    lowercase_race.str.contains("hispanic", na=False) &
    (df["is_black_derived"] == False)
)

df["is_white_derived"] = (
    lowercase_race.str.contains("white", na=False) &
    (lowercase_race.str.contains("hispanic", na=False) == False)
)

df["is_othereth_derived"] = (
    (df["is_black_derived"] == False) &
    (df["is_hisp_derived"] == False) &
    (df["is_white_derived"] == False)
)

#gender
df["is_male_derived"] = df["GENDER"] == "Male"

#age
df["age_derived"] = df["AGE_AT_INCIDENT"]
normal_age = df["age_derived"].quantile(.9999)
df.loc[df["age_derived"] > normal_age, "age_derived"] = normal_age

#datetime
df["sentenceymd_derived"] = df["SENTENCE_DATE"]

date_parts = df["sentenceymd_derived"].str.split("/", expand=True)
date_parts[2] = date_parts[2].apply(lambda x: "20" + str(x)[2:] if str(x).startswith("22") or str(x).startswith("29") else x)

df["sentenceymd_derived"] = date_parts[0] + "/" + date_parts[1] + "/" + date_parts[2]
df["sentenceymd_derived"] = pd.to_datetime(df["sentenceymd_derived"])

#month rounded
df["sentenceym_derived"] = df["sentenceymd_derived"].apply(lambda x: x.replace(day=1) if pd.notnull(x) else x)

#judge
df["SENTENCE_JUDGE_derived"] = df["SENTENCE_JUDGE"].astype(str)

df["SENTENCE_JUDGE_derived"] = df["SENTENCE_JUDGE_derived"].replace({
    "Doug Simpson": "Douglas Simpson",
    "D. Simpson": "Douglas Simpson",
    "Shelley Sutker-Kim": "Shelley Sutker",
    "S. Sutker": "Shelley Sutker"
})

judges = sorted(df["SENTENCE_JUDGE_derived"].unique())

judge_id_map = {}
for i, name in enumerate(judges):
    judge_id_map[name] = "judge_" + str(i + 1)

df["judgeid_derived"] = df["SENTENCE_JUDGE_derived"].map(judge_id_map)

#check
df.sample(n=10)[[
    "AGE_AT_INCIDENT", 
    "RACE", 
    "SENTENCE_DATE", 
    "SENTENCE_JUDGE",
    "age_derived", 
    "is_black_derived", 
    "is_hisp_derived", 
    "is_white_derived", 
    "is_othereth_derived",
    "sentenceymd_derived", 
    "sentenceym_derived", 
    "judgeid_derived"
]]



/var/folders/k9/0k40031s4q9btj5wvqclnww00000gn/T/ipykernel_40428/756167113.py:37: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["sentenceymd_derived"] = pd.to_datetime(df["sentenceymd_derived"])


,AGE_AT_INCIDENT,RACE,SENTENCE_DATE,SENTENCE_JUDGE,age_derived,is_black_derived,is_hisp_derived,is_white_derived,is_othereth_derived,sentenceymd_derived,sentenceym_derived,judgeid_derived
112860,35.0,Black,5/12/2014 12:00:00 AM,Mary Margaret Brosnahan,35.0,True,False,False,False,2014-05-12,2014-05-01,judge_160
166044,22.0,Black,6/1/2016 12:00:00 AM,Thomas M Davy,22.0,True,False,False,False,2016-06-01,2016-06-01,judge_240
97878,20.0,Black,6/27/2013 12:00:00 AM,Michael B McHale,20.0,True,False,False,False,2013-06-27,2013-06-01,judge_172
146654,61.0,Black,1/28/2015 12:00:00 AM,Thomas J Hennelly,61.0,True,False,False,False,2015-01-28,2015-01-01,judge_238
160942,43.0,Black,4/14/2017 12:00:00 AM,Catherine Marie Haberkorn,43.0,True,False,False,False,2017-04-14,2017-04-01,judge_31
117174,17.0,White,3/13/2014 12:00:00 AM,Colleen Ann Hyland,17.0,False,False,True,False,2014-03-13,2014-03-01,judge_36
195090,57.0,Black,1/26/2018 12:00:00 AM,Carol M Howard,57.0,True,False,False,False,2018-01-26,2018-01-01,judge_28
52856,42.0,White,3/7/2013 12:00:00 AM,Garritt E Howard,42.0,False,False,True,False,2013-03-07,2013-03-01,judge_68
193854,18.0,Black,1/10/2017 12:00:00 AM,URSULA WALOWSKI,18.0,True,False,False,False,2017-01-10,2017-01-01,judge_251
244232,NaN,Black,11/15/2019 12:00:00 AM,Ramon Ocasio,NaN,True,False,False,False,2019-11-15,2019-11-01,judge_198


## 1.4: Subsetting rows to analytic dataset (5 points)

You decide based on the above to simplify things in the following ways:
    
- Subset to cases where only one participant is charged, since cases with >1 participant might have complications like 
plea bargains/informing from other participants affecting the sentencing of the focal participant

- To go from a participant-case level dataset, where each participant is repeated across charges tied to the case, to a participant-level dataset, where each participant has one charge, subset to a participant's primary charge and their current sentence (`PRIMARY_CHARGE_FLAG` is True and `CURRENT_SENTENCE_FLAG` is True). Double check that this worked by confirming there are no longer multiple charges for the same case-participant

- Filter out observations where judge is nan or nonsensical (indicated by is.null or equal to FLOOD)

- Subset to sentencing date between 01-01-2012 and 04-05-2021 (inclusive)

After completing these steps, print the number of rows in the data

In [13]:
participants_per_case = df.groupby("CASE_ID")["CASE_PARTICIPANT_ID"].nunique()
one_person_cases = participants_per_case[participants_per_case == 1].index
df = df[df["CASE_ID"].isin(one_person_cases)]

df = df[df["PRIMARY_CHARGE_FLAG"] == True]
df = df[df["CURRENT_SENTENCE_FLAG"] == True]

df = df[df["SENTENCE_JUDGE"].notnull()]
df = df[df["SENTENCE_JUDGE"] != "FLOOD"]


start_date = pd.to_datetime("2012-01-01")
end_date = pd.to_datetime("2021-04-05")

df = df[df["sentenceymd_derived"] >= start_date]
df = df[df["sentenceymd_derived"] <= end_date]

print(len(df))

135162
